# 02 MLP

Self-contained MLP workflow. This notebook loads the common windows from preprocessing, extracts statistical features here, trains the MLP, evaluates it, and saves results.


## 1. MLP objective

The MLP does not use raw temporal windows. It receives one statistical feature vector per 30-second window.


## 2. Imports and configuration


In [ ]:
from pathlib import Path

cwd = Path.cwd()
if cwd.name == "notebooks":
    PROJECT_ROOT = cwd.parent
elif (cwd / "data").exists() and (cwd / "notebooks").exists():
    PROJECT_ROOT = cwd
elif (cwd / "wesad_stress_project").exists():
    PROJECT_ROOT = cwd / "wesad_stress_project"
else:
    PROJECT_ROOT = cwd
PROJECT_ROOT


## 3. Training and evaluation functions


In [ ]:
from __future__ import annotations

import json
import random
from pathlib import Path
from typing import Dict, Iterable, List, Tuple

import torch
import numpy as np
import pandas as pd
from sklearn.metrics import (
    average_precision_score,
    confusion_matrix,
    f1_score,
    precision_recall_fscore_support,
    roc_auc_score,
)


def set_seed(seed: int = 42) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


def logits_to_probabilities(logits: torch.Tensor) -> torch.Tensor:
    return torch.sigmoid(logits.reshape(-1))


def collect_probabilities(model: torch.nn.Module, loader, device: torch.device) -> Tuple[np.ndarray, np.ndarray]:
    model.eval()
    probabilities: List[np.ndarray] = []
    labels: List[np.ndarray] = []
    with torch.no_grad():
        for batch_x, batch_y in loader:
            batch_x = batch_x.to(device)
            logits = model(batch_x)
            probabilities.append(logits_to_probabilities(logits).cpu().numpy())
            labels.append(batch_y.float().cpu().numpy().reshape(-1))
    return np.concatenate(probabilities), np.concatenate(labels)


def select_threshold(y_true: np.ndarray, probabilities: np.ndarray) -> Tuple[float, pd.DataFrame]:
    rows = []
    for threshold in np.arange(0.10, 0.91, 0.01):
        predicted = (probabilities >= threshold).astype(int)
        rows.append(
            {
                "threshold": float(round(threshold, 2)),
                "macro_f1": float(f1_score(y_true, predicted, average="macro", zero_division=0)),
            }
        )
    table = pd.DataFrame(rows)
    best_row = table.sort_values(["macro_f1", "threshold"], ascending=[False, True]).iloc[0]
    return float(best_row["threshold"]), table


def binary_metrics(y_true: np.ndarray, probabilities: np.ndarray, threshold: float) -> Dict[str, object]:
    predicted = (probabilities >= threshold).astype(int)
    precision, recall, _, _ = precision_recall_fscore_support(
        y_true, predicted, labels=[0, 1], zero_division=0
    )
    metrics: Dict[str, object] = {
        "macro_f1": float(f1_score(y_true, predicted, average="macro", zero_division=0)),
        "weighted_f1": float(f1_score(y_true, predicted, average="weighted", zero_division=0)),
        "non_stress_precision": float(precision[0]),
        "non_stress_recall": float(recall[0]),
        "stress_precision": float(precision[1]),
        "stress_recall": float(recall[1]),
        "confusion_matrix": confusion_matrix(y_true, predicted, labels=[0, 1]).tolist(),
    }
    metrics["roc_auc"] = _safe_score(roc_auc_score, y_true, probabilities)
    metrics["average_precision"] = _safe_score(average_precision_score, y_true, probabilities)
    return metrics


def _safe_score(metric_fn, y_true: np.ndarray, probabilities: np.ndarray) -> float | None:
    try:
        return float(metric_fn(y_true, probabilities))
    except ValueError:
        return None


def per_subject_metrics(
    metadata: pd.DataFrame, y_true: np.ndarray, probabilities: np.ndarray, threshold: float
) -> pd.DataFrame:
    rows = []
    for subject_id, group in metadata.reset_index().groupby("subject_id"):
        indices = group["index"].to_numpy()
        metrics = binary_metrics(y_true[indices], probabilities[indices], threshold)
        metrics.pop("confusion_matrix", None)
        rows.append({"subject_id": subject_id, **metrics, "n_windows": int(len(indices))})
    return pd.DataFrame(rows)


def train_one_epoch(
    model: torch.nn.Module,
    loader,
    criterion,
    optimizer,
    device: torch.device,
    gradient_clip: float | None = None,
) -> float:
    model.train()
    losses = []
    for batch_x, batch_y in loader:
        batch_x = batch_x.to(device)
        batch_y = batch_y.float().to(device).reshape(-1)
        optimizer.zero_grad(set_to_none=True)
        logits = model(batch_x).reshape(-1)
        loss = criterion(logits, batch_y)
        loss.backward()
        if gradient_clip is not None:
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=gradient_clip)
        optimizer.step()
        losses.append(float(loss.detach().cpu()))
    return float(np.mean(losses))


def evaluate_loss(model: torch.nn.Module, loader, criterion, device: torch.device) -> float:
    model.eval()
    losses = []
    with torch.no_grad():
        for batch_x, batch_y in loader:
            batch_x = batch_x.to(device)
            batch_y = batch_y.float().to(device).reshape(-1)
            logits = model(batch_x).reshape(-1)
            loss = criterion(logits, batch_y)
            losses.append(float(loss.detach().cpu()))
    return float(np.mean(losses))


def train_with_early_stopping(
    model: torch.nn.Module,
    train_loader,
    validation_loader,
    criterion,
    optimizer,
    device: torch.device,
    max_epochs: int = 50,
    patience: int = 8,
    gradient_clip: float | None = None,
) -> Tuple[torch.nn.Module, pd.DataFrame, Dict[str, torch.Tensor]]:
    best_state = None
    best_validation_loss = float("inf")
    epochs_without_improvement = 0
    history = []

    for epoch in range(1, max_epochs + 1):
        train_loss = train_one_epoch(model, train_loader, criterion, optimizer, device, gradient_clip=gradient_clip)
        validation_loss = evaluate_loss(model, validation_loader, criterion, device)
        history.append(
            {
                "epoch": epoch,
                "train_loss": train_loss,
                "validation_loss": validation_loss,
            }
        )

        if validation_loss < best_validation_loss:
            best_validation_loss = validation_loss
            best_state = {key: value.detach().cpu().clone() for key, value in model.state_dict().items()}
            epochs_without_improvement = 0
        else:
            epochs_without_improvement += 1

        if epochs_without_improvement >= patience:
            break

    if best_state is not None:
        model.load_state_dict(best_state)
    return model, pd.DataFrame(history), best_state or {}


def pos_weight_from_labels(y_train: torch.Tensor, device: torch.device) -> torch.Tensor:
    y = y_train.reshape(-1)
    number_negative = (y == 0).sum().item()
    number_positive = (y == 1).sum().item()
    if number_positive == 0:
        raise ValueError("Cannot compute pos_weight because the training set has no positive labels.")
    return torch.tensor([number_negative / number_positive], dtype=torch.float32, device=device)


def count_parameters(model: torch.nn.Module) -> int:
    return sum(parameter.numel() for parameter in model.parameters() if parameter.requires_grad)


def save_json(path: Path, data: Dict[str, object]) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w", encoding="utf-8") as handle:
        json.dump(data, handle, indent=2)


def save_model_artifacts(
    artifact_dir: Path,
    model: torch.nn.Module,
    model_config: Dict[str, object],
    threshold: float,
    history: pd.DataFrame,
    validation_metrics: Dict[str, object],
    test_metrics: Dict[str, object],
    per_subject: pd.DataFrame,
    test_predictions: pd.DataFrame,
) -> None:
    artifact_dir.mkdir(parents=True, exist_ok=True)
    torch.save(model.state_dict(), artifact_dir / "best_model.pt")
    save_json(artifact_dir / "model_config.json", model_config)
    save_json(artifact_dir / "threshold.json", {"threshold": threshold})
    save_json(artifact_dir / "validation_metrics.json", validation_metrics)
    save_json(artifact_dir / "test_metrics.json", test_metrics)
    history.to_csv(artifact_dir / "training_history.csv", index=False)
    per_subject.to_csv(artifact_dir / "per_subject_metrics.csv", index=False)
    test_predictions.to_csv(artifact_dir / "test_predictions.csv", index=False)


def prediction_table(
    metadata: pd.DataFrame,
    y_true: np.ndarray,
    probabilities: np.ndarray,
    threshold: float,
) -> pd.DataFrame:
    return pd.DataFrame(
        {
            "window_id": metadata["window_id"].to_numpy(),
            "subject_id": metadata["subject_id"].to_numpy(),
            "true_label": y_true.astype(int),
            "stress_probability": probabilities,
            "predicted_label": (probabilities >= threshold).astype(int),
            "threshold": threshold,
        }
    )


In [ ]:
import pickle
import torch
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler
from torch.utils.data import DataLoader, TensorDataset

set_seed(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
BATCH_SIZE = 32
MAX_EPOCHS = 50
PATIENCE = 8
LEARNING_RATE = 1e-3
WEIGHT_DECAY = 1e-4
device


## 4. Load common windows and metadata

These are the same windows used by CNN, RNN, and LSTM.


In [ ]:
sequence_dir = PROJECT_ROOT / "data" / "processed" / "sequence"
metadata_dir = PROJECT_ROOT / "data" / "processed" / "metadata"

raw_paths = {
    "train": sequence_dir / "X_train_raw.pt",
    "validation": sequence_dir / "X_validation_raw.pt",
    "test": sequence_dir / "X_test_raw.pt",
}
missing_raw = [path.name for path in raw_paths.values() if not path.exists()]
if missing_raw:
    raise FileNotFoundError(
        "Raw sequence tensors are missing: "
        + ", ".join(missing_raw)
        + ". Rerun 01_preprocessing_and_splits.ipynb before this notebook."
    )

X_train_seq = torch.load(raw_paths["train"]).float()
y_train = torch.load(sequence_dir / "y_train.pt").float()
X_validation_seq = torch.load(raw_paths["validation"]).float()
y_validation = torch.load(sequence_dir / "y_validation.pt").float()
X_test_seq = torch.load(raw_paths["test"]).float()
y_test = torch.load(sequence_dir / "y_test.pt").float()

metadata_validation = pd.read_csv(metadata_dir / "windows_validation.csv")
metadata_test = pd.read_csv(metadata_dir / "windows_test.csv")

assert len(metadata_validation) == len(y_validation)
assert len(metadata_test) == len(y_test)
assert np.array_equal(metadata_validation["binary_label"].to_numpy(), y_validation.numpy().astype(int))
assert np.array_equal(metadata_test["binary_label"].to_numpy(), y_test.numpy().astype(int))
assert metadata_test["subject_id"].isin(["S2", "S11", "S14"]).all()

X_train_seq.shape, X_validation_seq.shape, X_test_seq.shape


## 5. Explain why an MLP needs statistical features

Flattening `(960, 6)` raw samples would discard temporal structure poorly and create a large input. Instead, each window is summarized with statistical, BVP, EDA, temperature, and movement features.


## 6. Define feature-extraction functions


In [ ]:
from __future__ import annotations

import json
import pickle
from dataclasses import dataclass
from pathlib import Path
from typing import Dict, Iterable, List, Tuple

import torch
import numpy as np
import pandas as pd
from scipy.signal import find_peaks, resample_poly
from scipy.stats import kurtosis, skew
from sklearn.preprocessing import StandardScaler


TRAIN_SUBJECTS = [
    "S3",
    "S4",
    "S6",
    "S7",
    "S8",
    "S9",
    "S10",
    "S13",
    "S16",
    "S17",
]
VALIDATION_SUBJECTS = ["S5", "S15"]
TEST_SUBJECTS = ["S2", "S11", "S14"]

SPLIT_SUBJECTS = {
    "train": TRAIN_SUBJECTS,
    "validation": VALIDATION_SUBJECTS,
    "test": TEST_SUBJECTS,
}

SEQUENCE_CHANNELS = ["BVP", "EDA", "TEMP", "ACC_x", "ACC_y", "ACC_z"]
ACCEPTED_LABELS = {1, 2, 3}
BINARY_LABEL_MAP = {1: 0, 2: 1, 3: 0}

TARGET_HZ = 32
WINDOW_SECONDS = 30
STRIDE_SECONDS = 15
WINDOW_SAMPLES = TARGET_HZ * WINDOW_SECONDS
STRIDE_SAMPLES = TARGET_HZ * STRIDE_SECONDS

WRIST_SAMPLE_RATES = {
    "BVP": 64,
    "EDA": 4,
    "TEMP": 4,
    "ACC": 32,
}
LABEL_HZ = 700


@dataclass(frozen=True)
class ProcessedSplit:
    X_sequence: np.ndarray
    y: np.ndarray
    X_mlp: np.ndarray
    metadata: pd.DataFrame


def load_subject_pickle(data_root: Path, subject_id: str) -> dict:
    path = data_root / subject_id / f"{subject_id}.pkl"
    with path.open("rb") as handle:
        return pickle.load(handle, encoding="latin1")


def _interp_to_target(values: np.ndarray, source_hz: int, target_times: np.ndarray) -> np.ndarray:
    values = np.asarray(values)
    if values.ndim == 1:
        values = values[:, None]

    source_times = np.arange(values.shape[0], dtype=np.float64) / float(source_hz)
    channels = [
        np.interp(target_times, source_times, values[:, channel]).astype(np.float32)
        for channel in range(values.shape[1])
    ]
    return np.column_stack(channels)


def _downsample_bvp_to_32hz(values: np.ndarray, n_samples: int) -> np.ndarray:
    values = np.asarray(values, dtype=np.float32).reshape(-1)
    downsampled = resample_poly(values, up=1, down=2).astype(np.float32)
    if downsampled.shape[0] < n_samples:
        source_times = np.arange(downsampled.shape[0], dtype=np.float64) / float(TARGET_HZ)
        target_times = np.arange(n_samples, dtype=np.float64) / float(TARGET_HZ)
        downsampled = np.interp(target_times, source_times, downsampled).astype(np.float32)
    return downsampled[:n_samples, None]


def align_subject_to_32hz(subject_data: dict) -> Tuple[np.ndarray, np.ndarray, np.ndarray]:
    wrist = subject_data["signal"]["wrist"]
    label = np.asarray(subject_data["label"]).reshape(-1)

    durations = [
        wrist["BVP"].shape[0] / WRIST_SAMPLE_RATES["BVP"],
        wrist["EDA"].shape[0] / WRIST_SAMPLE_RATES["EDA"],
        wrist["TEMP"].shape[0] / WRIST_SAMPLE_RATES["TEMP"],
        wrist["ACC"].shape[0] / WRIST_SAMPLE_RATES["ACC"],
        label.shape[0] / LABEL_HZ,
    ]
    duration = min(durations)
    n_samples = int(np.floor(duration * TARGET_HZ))
    target_times = np.arange(n_samples, dtype=np.float64) / float(TARGET_HZ)

    bvp = _downsample_bvp_to_32hz(wrist["BVP"], n_samples)
    eda = _interp_to_target(wrist["EDA"], WRIST_SAMPLE_RATES["EDA"], target_times)
    temp = _interp_to_target(wrist["TEMP"], WRIST_SAMPLE_RATES["TEMP"], target_times)
    acc = _interp_to_target(wrist["ACC"], WRIST_SAMPLE_RATES["ACC"], target_times)

    label_indices = np.minimum((target_times * LABEL_HZ).astype(np.int64), label.shape[0] - 1)
    aligned_labels = label[label_indices].astype(np.int16)
    aligned = np.column_stack([bvp[:, 0], eda[:, 0], temp[:, 0], acc]).astype(np.float32)
    return aligned, aligned_labels, target_times.astype(np.float32)


def continuous_label_segments(labels: np.ndarray, accepted_labels: Iterable[int] = ACCEPTED_LABELS) -> List[Tuple[int, int, int]]:
    accepted = set(accepted_labels)
    segments: List[Tuple[int, int, int]] = []
    start = None
    active_label = None

    for index, value in enumerate(labels):
        label = int(value)
        if label not in accepted:
            if start is not None:
                segments.append((start, index, active_label))
                start = None
                active_label = None
            continue

        if start is None:
            start = index
            active_label = label
        elif label != active_label:
            segments.append((start, index, active_label))
            start = index
            active_label = label

    if start is not None:
        segments.append((start, len(labels), active_label))

    return segments


def create_subject_windows(subject_id: str, split: str, subject_data: dict, start_window_id: int) -> Tuple[np.ndarray, np.ndarray, pd.DataFrame]:
    aligned, labels, times = align_subject_to_32hz(subject_data)
    windows = []
    y = []
    rows = []
    window_id = start_window_id

    for segment_start, segment_end, original_label in continuous_label_segments(labels):
        if segment_end - segment_start < WINDOW_SAMPLES:
            continue
        for start in range(segment_start, segment_end - WINDOW_SAMPLES + 1, STRIDE_SAMPLES):
            end = start + WINDOW_SAMPLES
            windows.append(aligned[start:end])
            y.append(BINARY_LABEL_MAP[original_label])
            rows.append(
                {
                    "window_id": window_id,
                    "subject_id": subject_id,
                    "split": split,
                    "start_time": float(times[start]),
                    "original_label": int(original_label),
                    "binary_label": int(BINARY_LABEL_MAP[original_label]),
                }
            )
            window_id += 1

    if not windows:
        return (
            np.empty((0, WINDOW_SAMPLES, len(SEQUENCE_CHANNELS)), dtype=np.float32),
            np.empty((0,), dtype=np.float32),
            pd.DataFrame(rows),
        )

    return np.stack(windows).astype(np.float32), np.asarray(y, dtype=np.float32), pd.DataFrame(rows)


def _general_features(values: np.ndarray, prefix: str) -> Dict[str, float]:
    x = np.asarray(values, dtype=np.float64)
    t = np.arange(x.size, dtype=np.float64) / TARGET_HZ
    slope = np.polyfit(t, x, 1)[0] if x.size > 1 else 0.0
    q25, q75 = np.percentile(x, [25, 75])
    signal_std = float(np.std(x))
    if signal_std < 1e-8:
        skewness_value = 0.0
        kurtosis_value = 0.0
    else:
        skewness_value = float(skew(x, bias=False, nan_policy="omit"))
        kurtosis_value = float(kurtosis(x, bias=False, nan_policy="omit"))
    return {
        f"{prefix}_mean": float(np.mean(x)),
        f"{prefix}_std": signal_std,
        f"{prefix}_min": float(np.min(x)),
        f"{prefix}_max": float(np.max(x)),
        f"{prefix}_median": float(np.median(x)),
        f"{prefix}_p25": float(q25),
        f"{prefix}_p75": float(q75),
        f"{prefix}_iqr": float(q75 - q25),
        f"{prefix}_range": float(np.max(x) - np.min(x)),
        f"{prefix}_rms": float(np.sqrt(np.mean(np.square(x)))),
        f"{prefix}_energy": float(np.sum(np.square(x))),
        f"{prefix}_slope": float(slope),
        f"{prefix}_skewness": skewness_value,
        f"{prefix}_kurtosis": kurtosis_value,
    }


def _safe_mean(values: np.ndarray) -> float:
    return float(np.mean(values)) if len(values) else 0.0


def _bvp_features(bvp: np.ndarray) -> Dict[str, float]:
    distance = max(1, int(0.35 * TARGET_HZ))
    prominence = max(float(np.std(bvp)) * 0.25, 1e-6)
    peaks, _ = find_peaks(bvp, distance=distance, prominence=prominence)
    peak_times = peaks / TARGET_HZ
    ibi = np.diff(peak_times)
    valid_ibi = ibi[(ibi >= 0.3) & (ibi <= 2.0)]
    hr = 60.0 / valid_ibi if len(valid_ibi) else np.array([])
    rmssd = np.sqrt(np.mean(np.square(np.diff(valid_ibi)))) if len(valid_ibi) > 1 else 0.0
    return {
        "BVP_peak_count": float(len(peaks)),
        "BVP_mean_peak_amplitude": _safe_mean(bvp[peaks]) if len(peaks) else 0.0,
        "BVP_mean_heart_rate": _safe_mean(hr),
        "BVP_mean_ibi": _safe_mean(valid_ibi),
        "BVP_ibi_std": float(np.std(valid_ibi)) if len(valid_ibi) else 0.0,
        "BVP_rmssd": float(rmssd),
    }


def _eda_features(eda: np.ndarray) -> Dict[str, float]:
    prominence = max(float(np.std(eda)) * 0.2, 1e-6)
    peaks, props = find_peaks(eda, distance=int(1.0 * TARGET_HZ), prominence=prominence)
    amplitudes = props.get("prominences", np.array([]))
    slope = np.polyfit(np.arange(eda.size) / TARGET_HZ, eda, 1)[0]
    return {
        "EDA_specific_mean": float(np.mean(eda)),
        "EDA_specific_slope": float(slope),
        "EDA_specific_range": float(np.max(eda) - np.min(eda)),
        "EDA_response_count": float(len(peaks)),
        "EDA_mean_response_amplitude": _safe_mean(amplitudes),
        "EDA_max_response_amplitude": float(np.max(amplitudes)) if len(amplitudes) else 0.0,
    }


def _temperature_features(temp: np.ndarray) -> Dict[str, float]:
    slope = np.polyfit(np.arange(temp.size) / TARGET_HZ, temp, 1)[0]
    return {
        "TEMP_specific_mean": float(np.mean(temp)),
        "TEMP_specific_range": float(np.max(temp) - np.min(temp)),
        "TEMP_specific_slope": float(slope),
        "TEMP_final_minus_initial": float(temp[-1] - temp[0]),
    }


def _acc_features(acc: np.ndarray) -> Dict[str, float]:
    magnitude = np.linalg.norm(acc, axis=1)
    jerk = np.diff(magnitude) * TARGET_HZ
    return {
        "ACC_magnitude_mean": float(np.mean(magnitude)),
        "ACC_magnitude_std": float(np.std(magnitude)),
        "ACC_movement_energy": float(np.sum(np.square(magnitude))),
        "ACC_jerk_mean": _safe_mean(jerk),
        "ACC_jerk_std": float(np.std(jerk)) if len(jerk) else 0.0,
    }


def extract_window_features(window: np.ndarray) -> Dict[str, float]:
    features: Dict[str, float] = {}
    for channel_index, channel_name in enumerate(SEQUENCE_CHANNELS):
        features.update(_general_features(window[:, channel_index], channel_name))
    features.update(_bvp_features(window[:, 0]))
    features.update(_eda_features(window[:, 1]))
    features.update(_temperature_features(window[:, 2]))
    features.update(_acc_features(window[:, 3:6]))
    return {key: (0.0 if not np.isfinite(value) else float(value)) for key, value in features.items()}


def extract_feature_table(X: np.ndarray) -> pd.DataFrame:
    if len(X) == 0:
        return pd.DataFrame()
    return pd.DataFrame([extract_window_features(window) for window in X])


def _fit_transform_sequences(
    X_train: np.ndarray, splits: Dict[str, np.ndarray]
) -> Tuple[Dict[str, np.ndarray], StandardScaler]:
    scaler = StandardScaler()
    scaler.fit(X_train.reshape(-1, X_train.shape[-1]))
    transformed = {}
    for split, X in splits.items():
        transformed[split] = scaler.transform(X.reshape(-1, X.shape[-1])).reshape(X.shape).astype(np.float32)
    return transformed, scaler


def _fit_transform_features(
    train_features: pd.DataFrame, feature_tables: Dict[str, pd.DataFrame]
) -> Tuple[Dict[str, np.ndarray], StandardScaler, List[str]]:
    feature_columns = list(train_features.columns)
    scaler = StandardScaler()
    scaler.fit(train_features[feature_columns])
    transformed = {
        split: scaler.transform(table[feature_columns]).astype(np.float32)
        for split, table in feature_tables.items()
    }
    return transformed, scaler, feature_columns


def run_preprocessing(project_root: Path) -> Dict[str, object]:
    data_root = project_root / "data" / "WESAD" / "WESAD"
    sequence_dir = project_root / "data" / "processed" / "sequence"
    feature_dir = project_root / "data" / "processed" / "features"
    metadata_dir = project_root / "data" / "processed" / "metadata"
    artifact_dir = project_root / "artifacts" / "preprocessing"
    for directory in [sequence_dir, feature_dir, metadata_dir, artifact_dir]:
        directory.mkdir(parents=True, exist_ok=True)

    raw_sequences: Dict[str, List[np.ndarray]] = {split: [] for split in SPLIT_SUBJECTS}
    labels: Dict[str, List[np.ndarray]] = {split: [] for split in SPLIT_SUBJECTS}
    metadata_frames: Dict[str, List[pd.DataFrame]] = {split: [] for split in SPLIT_SUBJECTS}
    next_window_id = 0

    for split, subjects in SPLIT_SUBJECTS.items():
        for subject_id in subjects:
            subject_data = load_subject_pickle(data_root, subject_id)
            X_subject, y_subject, meta_subject = create_subject_windows(
                subject_id, split, subject_data, next_window_id
            )
            if len(meta_subject):
                next_window_id = int(meta_subject["window_id"].max()) + 1
            raw_sequences[split].append(X_subject)
            labels[split].append(y_subject)
            metadata_frames[split].append(meta_subject)

    X_raw = {
        split: np.concatenate(parts, axis=0).astype(np.float32)
        for split, parts in raw_sequences.items()
    }
    y = {
        split: np.concatenate(parts, axis=0).astype(np.float32)
        for split, parts in labels.items()
    }
    metadata = {
        split: pd.concat(parts, ignore_index=True)
        for split, parts in metadata_frames.items()
    }

    assert set(TRAIN_SUBJECTS).isdisjoint(VALIDATION_SUBJECTS)
    assert set(TRAIN_SUBJECTS).isdisjoint(TEST_SUBJECTS)
    assert set(VALIDATION_SUBJECTS).isdisjoint(TEST_SUBJECTS)

    X_sequence, sequence_scaler = _fit_transform_sequences(X_raw["train"], X_raw)

    feature_tables = {split: extract_feature_table(X_raw[split]) for split in SPLIT_SUBJECTS}
    X_mlp, feature_scaler, feature_columns = _fit_transform_features(feature_tables["train"], feature_tables)

    for split in SPLIT_SUBJECTS:
        assert X_raw[split].shape[1:] == (WINDOW_SAMPLES, len(SEQUENCE_CHANNELS))
        assert X_sequence[split].shape[1:] == (WINDOW_SAMPLES, len(SEQUENCE_CHANNELS))
        assert np.isfinite(X_raw[split]).all()
        assert np.isfinite(X_sequence[split]).all()
        assert set(np.unique(y[split]).astype(float).tolist()) == {0.0, 1.0}
        assert len(metadata[split]) == len(y[split])
        assert np.array_equal(metadata[split]["binary_label"].to_numpy(), y[split].astype(int))

        torch.save(torch.from_numpy(X_raw[split]), sequence_dir / f"X_{split}_raw.pt")
        torch.save(torch.from_numpy(X_sequence[split]), sequence_dir / f"X_{split}.pt")
        torch.save(torch.from_numpy(y[split]), sequence_dir / f"y_{split}.pt")
        metadata[split].to_csv(metadata_dir / f"windows_{split}.csv", index=False)

        pd.DataFrame(X_mlp[split], columns=feature_columns).to_csv(feature_dir / f"X_{split}.csv", index=False)
        pd.Series(y[split], name="binary_label").to_csv(feature_dir / f"y_{split}.csv", index=False)

    all_metadata = pd.concat(metadata.values(), ignore_index=True)
    all_metadata.to_csv(metadata_dir / "windows_all.csv", index=False)

    with (artifact_dir / "feature_columns.json").open("w", encoding="utf-8") as handle:
        json.dump(feature_columns, handle, indent=2)
    with (artifact_dir / "sequence_channels.json").open("w", encoding="utf-8") as handle:
        json.dump(SEQUENCE_CHANNELS, handle, indent=2)
    with (artifact_dir / "split_subjects.json").open("w", encoding="utf-8") as handle:
        json.dump(SPLIT_SUBJECTS, handle, indent=2)
    with (artifact_dir / "preprocessing_config.json").open("w", encoding="utf-8") as handle:
        json.dump(
            {
                "target_hz": TARGET_HZ,
                "window_seconds": WINDOW_SECONDS,
                "window_samples": WINDOW_SAMPLES,
                "stride_seconds": STRIDE_SECONDS,
                "stride_samples": STRIDE_SAMPLES,
                "sequence_channels": SEQUENCE_CHANNELS,
                "accepted_labels": sorted(ACCEPTED_LABELS),
                "binary_label_map": BINARY_LABEL_MAP,
            },
            handle,
            indent=2,
        )

    np.savez(
        artifact_dir / "sequence_scaler.npz",
        mean=sequence_scaler.mean_,
        scale=sequence_scaler.scale_,
        var=sequence_scaler.var_,
    )
    np.savez(
        artifact_dir / "feature_scaler.npz",
        mean=feature_scaler.mean_,
        scale=feature_scaler.scale_,
        var=feature_scaler.var_,
    )
    with (artifact_dir / "feature_scaler.pkl").open("wb") as handle:
        pickle.dump(feature_scaler, handle)

    return {
        "sequence_shapes": {split: tuple(X_sequence[split].shape) for split in SPLIT_SUBJECTS},
        "feature_shapes": {split: tuple(X_mlp[split].shape) for split in SPLIT_SUBJECTS},
        "label_counts": {
            split: pd.Series(y[split]).value_counts().sort_index().to_dict()
            for split in SPLIT_SUBJECTS
        },
        "metadata_rows": {split: int(len(metadata[split])) for split in SPLIT_SUBJECTS},
        "feature_columns": feature_columns,
    }


## 7. Extract features from training windows


In [ ]:
X_train_features_df = extract_feature_table(X_train_seq.numpy())
X_train_features_df.shape


## 8. Extract features from validation and test windows


In [ ]:
X_validation_features_df = extract_feature_table(X_validation_seq.numpy())
X_test_features_df = extract_feature_table(X_test_seq.numpy())
X_validation_features_df.shape, X_test_features_df.shape


## 9. Fit feature scaler using training features only


In [ ]:
candidate_columns = list(X_train_features_df.columns)
exact_duplicate_mask = X_train_features_df[candidate_columns].T.duplicated()
exact_duplicate_columns = [
    column for column, is_duplicate in zip(candidate_columns, exact_duplicate_mask) if is_duplicate
]
candidate_columns = [column for column in candidate_columns if column not in exact_duplicate_columns]

correlation_matrix = X_train_features_df[candidate_columns].corr().abs()
upper_triangle = correlation_matrix.where(
    np.triu(np.ones(correlation_matrix.shape), k=1).astype(bool)
)
high_correlation_columns = [
    column for column in upper_triangle.columns if (upper_triangle[column] > 0.95).any()
]
feature_columns = [column for column in candidate_columns if column not in high_correlation_columns]

feature_scaler = StandardScaler()
feature_scaler.fit(X_train_features_df[feature_columns])

X_train = torch.tensor(feature_scaler.transform(X_train_features_df[feature_columns]), dtype=torch.float32)
X_validation = torch.tensor(feature_scaler.transform(X_validation_features_df[feature_columns]), dtype=torch.float32)
X_test = torch.tensor(feature_scaler.transform(X_test_features_df[feature_columns]), dtype=torch.float32)

artifact_dir = PROJECT_ROOT / "artifacts" / "preprocessing"
artifact_dir.mkdir(parents=True, exist_ok=True)
with (artifact_dir / "feature_columns.json").open("w", encoding="utf-8") as handle:
    json.dump(feature_columns, handle, indent=2)
with (artifact_dir / "feature_scaler.pkl").open("wb") as handle:
    pickle.dump(feature_scaler, handle)
with (artifact_dir / "feature_selection.json").open("w", encoding="utf-8") as handle:
    json.dump(
        {
            "removed_exact_duplicate_columns": exact_duplicate_columns,
            "removed_high_correlation_columns": high_correlation_columns,
            "correlation_threshold": 0.95,
        },
        handle,
        indent=2,
    )

feature_dir = PROJECT_ROOT / "data" / "processed" / "features"
feature_dir.mkdir(parents=True, exist_ok=True)
X_train_features_df.to_csv(feature_dir / "X_train_unscaled.csv", index=False)
X_validation_features_df.to_csv(feature_dir / "X_validation_unscaled.csv", index=False)
X_test_features_df.to_csv(feature_dir / "X_test_unscaled.csv", index=False)
pd.DataFrame(X_train.numpy(), columns=feature_columns).to_csv(feature_dir / "X_train.csv", index=False)
pd.DataFrame(X_validation.numpy(), columns=feature_columns).to_csv(feature_dir / "X_validation.csv", index=False)
pd.DataFrame(X_test.numpy(), columns=feature_columns).to_csv(feature_dir / "X_test.csv", index=False)
X_train.shape, X_validation.shape, X_test.shape, len(exact_duplicate_columns), len(high_correlation_columns)


## 10. Create PyTorch datasets and DataLoaders


In [ ]:
train_dataset = TensorDataset(X_train, y_train)
validation_dataset = TensorDataset(X_validation, y_validation)
test_dataset = TensorDataset(X_test, y_test)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
validation_loader = DataLoader(validation_dataset, batch_size=BATCH_SIZE, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)


## 11. Define the MLP class


In [ ]:
from __future__ import annotations

import torch
from torch import nn


class MLPClassifier(nn.Module):
    def __init__(self, input_dim: int, dropout: float = 0.3) -> None:
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 128),
            nn.ReLU(),
            nn.BatchNorm1d(128),
            nn.Dropout(dropout),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.BatchNorm1d(64),
            nn.Dropout(dropout),
            nn.Linear(64, 1),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.net(x)


## 12. Display model architecture


In [ ]:
model = MLPClassifier(input_dim=X_train.shape[1]).to(device)
model


## 13. Define training loop


Training utilities above use raw logits with `BCEWithLogitsLoss`. The final model layer does not include a sigmoid.


## 14. Define validation loop


Validation collects probabilities with `torch.sigmoid(logits)` and uses validation data only to select a threshold.


## 15. Train without imbalance correction


In [ ]:
set_seed(42)
unweighted_model = MLPClassifier(input_dim=X_train.shape[1]).to(device)
unweighted_criterion = torch.nn.BCEWithLogitsLoss()
unweighted_optimizer = torch.optim.Adam(unweighted_model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
unweighted_model, unweighted_history, _ = train_with_early_stopping(
    unweighted_model,
    train_loader,
    validation_loader,
    unweighted_criterion,
    unweighted_optimizer,
    device,
    max_epochs=MAX_EPOCHS,
    patience=PATIENCE,
)
unweighted_history.tail()


## 16. Train with class weights


In [ ]:
set_seed(42)
model = MLPClassifier(input_dim=X_train.shape[1]).to(device)
pos_weight = pos_weight_from_labels(y_train, device)
criterion = torch.nn.BCEWithLogitsLoss(pos_weight=pos_weight)
optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)

model, history, best_state = train_with_early_stopping(
    model,
    train_loader,
    validation_loader,
    criterion,
    optimizer,
    device,
    max_epochs=MAX_EPOCHS,
    patience=PATIENCE,
)
history.tail()


## 17. Select threshold using validation data


In [ ]:
unweighted_validation_probabilities, unweighted_validation_true = collect_probabilities(unweighted_model, validation_loader, device)
unweighted_threshold, _ = select_threshold(unweighted_validation_true, unweighted_validation_probabilities)
unweighted_validation_metrics = binary_metrics(
    unweighted_validation_true,
    unweighted_validation_probabilities,
    unweighted_threshold,
)

weighted_validation_probabilities, weighted_validation_true = collect_probabilities(model, validation_loader, device)
weighted_threshold, _ = select_threshold(weighted_validation_true, weighted_validation_probabilities)
weighted_validation_metrics = binary_metrics(
    weighted_validation_true,
    weighted_validation_probabilities,
    weighted_threshold,
)

unweighted_best_epoch = int(unweighted_history.loc[unweighted_history["validation_loss"].idxmin(), "epoch"])
weighted_best_epoch = int(history.loc[history["validation_loss"].idxmin(), "epoch"])

validation_variant_comparison = pd.DataFrame(
    [
        {
            "method": "no_correction",
            "best_epoch": unweighted_best_epoch,
            "threshold": unweighted_threshold,
            "macro_f1": unweighted_validation_metrics["macro_f1"],
            "stress_recall": unweighted_validation_metrics["stress_recall"],
            "average_precision": unweighted_validation_metrics["average_precision"],
        },
        {
            "method": "class_weight",
            "best_epoch": weighted_best_epoch,
            "threshold": weighted_threshold,
            "macro_f1": weighted_validation_metrics["macro_f1"],
            "stress_recall": weighted_validation_metrics["stress_recall"],
            "average_precision": weighted_validation_metrics["average_precision"],
        },
    ]
)
display(validation_variant_comparison)

if weighted_validation_metrics["macro_f1"] >= unweighted_validation_metrics["macro_f1"]:
    selected_model = model
    selected_history = history
    selected_threshold = weighted_threshold
    selected_validation_probabilities = weighted_validation_probabilities
    selected_validation_true = weighted_validation_true
    selected_validation_metrics = weighted_validation_metrics
    selected_imbalance_method = "class_weight"
    selected_best_epoch = weighted_best_epoch
else:
    selected_model = unweighted_model
    selected_history = unweighted_history
    selected_threshold = unweighted_threshold
    selected_validation_probabilities = unweighted_validation_probabilities
    selected_validation_true = unweighted_validation_true
    selected_validation_metrics = unweighted_validation_metrics
    selected_imbalance_method = "no_correction"
    selected_best_epoch = unweighted_best_epoch

model = selected_model
history = selected_history
threshold = selected_threshold
validation_probabilities = selected_validation_probabilities
validation_true = selected_validation_true
validation_metrics = {
    **selected_validation_metrics,
    "selected_imbalance_method": selected_imbalance_method,
    "best_validation_epoch": selected_best_epoch,
    "variant_comparison": validation_variant_comparison.to_dict(orient="records"),
}

threshold, selected_imbalance_method, validation_metrics


## 18. Evaluate test data


In [ ]:
test_probabilities, test_true = collect_probabilities(model, test_loader, device)
test_metrics = binary_metrics(test_true, test_probabilities, threshold)
test_metrics


## 19. Evaluate by subject


In [ ]:
subject_metrics = per_subject_metrics(metadata_test, test_true, test_probabilities, threshold)
subject_metrics


## 20. Plot confusion matrix


In [ ]:
cm = np.array(test_metrics["confusion_matrix"])
fig, ax = plt.subplots(figsize=(4, 4))
im = ax.imshow(cm, cmap="Blues")
ax.set_xticks([0, 1], labels=["non-stress", "stress"])
ax.set_yticks([0, 1], labels=["non-stress", "stress"])
ax.set_xlabel("Predicted")
ax.set_ylabel("True")
for i in range(2):
    for j in range(2):
        ax.text(j, i, cm[i, j], ha="center", va="center")
fig.colorbar(im, ax=ax)
plt.show()


## 21. Plot ROC and precision-recall curves


In [ ]:
from sklearn.metrics import PrecisionRecallDisplay, RocCurveDisplay

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
RocCurveDisplay.from_predictions(test_true, test_probabilities, ax=axes[0])
PrecisionRecallDisplay.from_predictions(test_true, test_probabilities, ax=axes[1])
plt.tight_layout()
plt.show()


## 22. SHAP analysis

SHAP explains model behavior, not physiological causality.


In [ ]:
try:
    import shap

    model.eval()
    shap_model = model.to("cpu")
    background = X_train[: min(100, len(X_train))]
    explain_count = min(200, len(X_validation))
    samples = X_validation[:explain_count]

    with torch.no_grad():
        print("Model output shape for SHAP:", shap_model(background[:4]).shape)

    explainer = shap.DeepExplainer(shap_model, background)
    shap_values = explainer.shap_values(samples)
    values = shap_values[0] if isinstance(shap_values, list) else shap_values
    values = np.asarray(values).squeeze()

    shap.summary_plot(values, samples.numpy(), feature_names=feature_columns, show=True)
    shap.summary_plot(values, samples.numpy(), feature_names=feature_columns, plot_type="bar", show=True)

    probs = validation_probabilities[:explain_count]
    for title, index in [("highest stress-score example", int(np.argmax(probs))), ("lowest stress-score example", int(np.argmin(probs)))]:
        explanation = shap.Explanation(
            values=values[index],
            base_values=np.asarray(explainer.expected_value).reshape(-1)[0],
            data=samples[index].numpy(),
            feature_names=feature_columns,
        )
        shap.plots.waterfall(explanation, max_display=15, show=True)

    model.to(device)
except ImportError:
    print("Install shap to run SHAP explanations: pip install shap")


## 23. Save model and results


In [ ]:
artifact_dir = PROJECT_ROOT / "artifacts" / "models" / "mlp"
test_predictions = prediction_table(metadata_test, test_true, test_probabilities, threshold)

save_model_artifacts(
    artifact_dir=artifact_dir,
    model=model,
    model_config={
        "model": "MLPClassifier",
        "input_dim": int(X_train.shape[1]),
        "dropout": 0.3,
        "parameter_count": count_parameters(model),
        "selected_imbalance_method": selected_imbalance_method,
        "best_validation_epoch": selected_best_epoch,
        "random_seed": 42,
        "uses_class_weighting": selected_imbalance_method == "class_weight",
        "input_representation": "statistical_features_extracted_from_raw_windows",
        "removed_exact_duplicate_features": exact_duplicate_columns,
        "removed_high_correlation_features": high_correlation_columns,
    },
    threshold=threshold,
    history=history,
    validation_metrics=validation_metrics,
    test_metrics=test_metrics,
    per_subject=subject_metrics,
    test_predictions=test_predictions,
)
artifact_dir


## 24. Conclusion

The MLP is evaluated on the same windows and subjects as the sequence models, but it receives statistical features instead of raw temporal sequences.
